# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [3]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6
        self.start = (0, 0)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}
        self.terminal_states = {
            (0, 5): 10.0,  # entrega
            (2, 2): 2.0,   # estación de carga
            (3, 5): -10.0, # peligro mortal
        }
        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }
        self.living_reward = -1.0
        self.gamma = 0.9
        self.actions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    def is_valid_state(self, state):
        r, c = state
        return 0 <= r < self.height and 0 <= c < self.width and state not in self.walls

    def states(self):
        return [(r, c) for r in range(self.height) for c in range(self.width)
                if (r, c) not in self.walls]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        if self.is_terminal(state):
            return [(state, 1.0)]
        dr, dc = action
        left = (-dc, dr)
        right = (dc, -dr)
        intended, side = (0.60, 0.20) if state in self.slippery_states else (0.90, 0.05)
        outcomes = [(action, intended), (left, side), (right, side)]
        probs = {}
        for (drow, dcol), p in outcomes:
            nxt = (state[0] + drow, state[1] + dcol)
            if not self.is_valid_state(nxt):
                nxt = state
            probs[nxt] = probs.get(nxt, 0.0) + p
        return list(probs.items())



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [4]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [5]:
def expected_next_value(grid, state, action, V):
    return sum(prob * V[next_state]
               for next_state, prob in grid.get_transition_probs(state, action))


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {s: (grid.get_reward(s) if grid.is_terminal(s) else 0.0) for s in grid.states()}
    for iteration in range(1, max_iter + 1):
        delta = 0.0
        new_V = V.copy()
        for s in grid.states():
            if grid.is_terminal(s):
                new_V[s] = grid.get_reward(s)
            else:
                new_V[s] = grid.get_reward(s) + grid.gamma * max(
                    expected_next_value(grid, s, a, V) for a in grid.actions)
            delta = max(delta, abs(new_V[s] - V[s]))
        V = new_V
        if delta < threshold:
            return V, iteration
    return V, max_iter


def extract_policy(grid, V):
    policy = {}
    for s in grid.states():
        if not grid.is_terminal(s):
            policy[s] = max(grid.actions,
                            key=lambda a: expected_next_value(grid, s, a, V))
    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [6]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {s: (grid.get_reward(s) if grid.is_terminal(s) else 0.0) for s in grid.states()}
    for _ in range(max_iter):
        delta = 0.0
        new_V = V.copy()
        for s in grid.states():
            if grid.is_terminal(s):
                new_V[s] = grid.get_reward(s)
            else:
                new_V[s] = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, policy[s], V)
            delta = max(delta, abs(new_V[s] - V[s]))
        V = new_V
        if delta < threshold:
            break
    return V


def policy_improvement(grid, V):
    return {s: max(grid.actions, key=lambda a: expected_next_value(grid, s, a, V))
            for s in grid.states() if not grid.is_terminal(s)}


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    policy = {s: grid.actions[0] for s in grid.states() if not grid.is_terminal(s)}
    history = []
    for _ in range(max_iter):
        V = policy_evaluation(grid, policy, threshold=threshold)
        improved = policy_improvement(grid, V)
        history.append(sum(policy[s] != improved[s] for s in policy))
        if all(policy[s] == improved[s] for s in policy):
            return improved, V, history
        policy = improved
    return policy, policy_evaluation(grid, policy, threshold=threshold), history



## Parte 4 — Visualización y comparación


In [7]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [8]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 19

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [16, 5, 1, 0]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


### Respuestas — Parte 5

**1. Desde `START`, ¿el robot busca la entrega +10 o prefiere la estación de carga +2?**

Con los parámetros originales (`living_reward = -1` y `gamma = 0.9`), la política óptima elige **derecha (→)** desde `START`. Por tanto, intenta avanzar por la fila superior hacia la entrega `+10`, aunque la estantería obliga a rodear el obstáculo. La estación de carga `+2` no es la primera elección de la política en el estado inicial.

**2. ¿Por qué una recompensa menor podría ser óptima?**

Porque el objetivo no es elegir la recompensa inmediata más alta, sino maximizar la suma de recompensas futuras descontadas. Una recompensa `+2` puede ser preferible si se alcanza con menos pasos y menos riesgo que la entrega `+10`. El costo por paso y la incertidumbre de resbalar pueden hacer que una ruta larga tenga menor retorno esperado.

**3. ¿En qué estados el piso resbaloso cambia la decisión?**

El modelo marca como resbalosos `(1, 2)`, `(2, 1)` y `(3, 3)`. En esos estados la acción elegida ocurre con probabilidad `0.60` y cada desviación lateral con `0.20`, en lugar de `0.90` y `0.05` en piso normal. Eso cambia los valores esperados de las acciones y puede cambiar la política en esos estados y en estados cercanos. El cambio concreto se debe comprobar comparando las políticas impresas.

**4. ¿Qué papel cumple el costo por paso `-1`?**

Penaliza cada estado transitable no terminal. Así, el robot tiene un incentivo para evitar recorridos innecesariamente largos y llegar pronto a un terminal conveniente. Si el costo se reduce, explorar rutas más largas hacia una recompensa grande puede resultar más atractivo.

**5. ¿Por qué `T(s,a,s')` no puede implementarse con las mismas probabilidades para todos los estados?**

Porque la probabilidad depende del tipo de piso del estado actual: en piso normal el movimiento intencionado tiene probabilidad `0.90`, mientras que en piso resbaloso tiene probabilidad `0.60`. Además, cuando una acción sale del mapa o choca con una estantería, el robot permanece en el mismo estado. Por eso la transición depende de `s`, `a` y los límites/obstáculos del mapa.


### Experimento A — Menos costo por paso

**Predicción antes de ejecutar:** al cambiar `living_reward` de `-1.0` a `-0.1`, cada paso cuesta menos. Por eso espero que el robot esté más dispuesto a recorrer una ruta larga para alcanzar `+10`. Ejecuta la celda siguiente y compara la acción de `START` y la tabla de política con el caso original.


In [9]:
# Experimento A: menor costo por paso
grid_A = WarehouseMDP()
grid_A.living_reward = -0.1

V_A, n_A = value_iteration(grid_A)
pi_A = extract_policy(grid_A, V_A)

print("=== EXPERIMENTO A ===")
print("living_reward:", grid_A.living_reward)
print("Iteraciones:", n_A)
print("Valor de START:", round(V_A[grid_A.start], 4))
print("Acción desde START:", ARROWS[pi_A[grid_A.start]])
print_policy(grid_A, pi_A)


=== EXPERIMENTO A ===
living_reward: -0.1
Iteraciones: 25
Valor de START: 1.6487
Acción desde START: →
 →  |  →  |  ↓  |  #  |  →  | +10
 ↑  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  →  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 


**Interpretación A:** el costo por paso menos negativo hace que el retorno de las rutas largas mejore. Compara la política obtenida con la política original; no supongas que la política cambia en todas las celdas, porque el cambio depende de los valores esperados de las rutas.


### Experimento B — Piso muy resbaloso

**Predicción antes de ejecutar:** al bajar la probabilidad de movimiento intencionado de `0.60` a `0.40`, el robot pierde precisión en las celdas resbalosas. Las desviaciones laterales suben a `0.30` cada una. Espero que las rutas que atraviesan esas celdas sean menos atractivas o que la política cambie para evitar parte de ese riesgo.


In [10]:
# Experimento B: movimiento intencionado 0.40 y desviaciones 0.30 + 0.30
grid_B = WarehouseMDP()

def transition_probs_slippery_040(state, action):
    if grid_B.is_terminal(state):
        return [(state, 1.0)]

    dr, dc = action
    left = (-dc, dr)
    right = (dc, -dr)

    if state in grid_B.slippery_states:
        intended, side = 0.40, 0.30
    else:
        intended, side = 0.90, 0.05

    probs = {}
    for (drow, dcol), p in [(action, intended), (left, side), (right, side)]:
        nxt = (state[0] + drow, state[1] + dcol)
        if not grid_B.is_valid_state(nxt):
            nxt = state
        probs[nxt] = probs.get(nxt, 0.0) + p
    return list(probs.items())

grid_B.get_transition_probs = transition_probs_slippery_040

# Verificación: todas las distribuciones siguen sumando 1.
for s in grid_B.states():
    for a in grid_B.actions:
        assert abs(sum(p for _, p in grid_B.get_transition_probs(s, a)) - 1.0) < 1e-12

V_B, n_B = value_iteration(grid_B)
pi_B = extract_policy(grid_B, V_B)

print("=== EXPERIMENTO B ===")
print("Probabilidad intencionada en piso resbaloso: 0.40")
print("Probabilidad de cada desviación: 0.30")
print("Iteraciones:", n_B)
print("Valor de START:", round(V_B[grid_B.start], 4))
print("Acción desde START:", ARROWS[pi_B[grid_B.start]])
print_policy(grid_B, pi_B)


=== EXPERIMENTO B ===
Probabilidad intencionada en piso resbaloso: 0.40
Probabilidad de cada desviación: 0.30
Iteraciones: 20
Valor de START: -2.7062
Acción desde START: →
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 


**Interpretación B:** la distribución sigue sumando 1 (`0.40 + 0.30 + 0.30 = 1`). El efecto sobre la política debe evaluarse comparando la tabla de este experimento con la política original. El modelo representa el riesgo adicional; no garantiza que cambie la acción en `START`, porque el cambio puede afectar principalmente estados más cercanos al piso resbaloso.


### Experimento C — Más paciencia

**Predicción antes de ejecutar:** al aumentar `gamma` de `0.9` a `0.99`, las recompensas futuras se descuentan menos. Espero que la entrega `+10`, si requiere más pasos, gane atractivo relativo frente a una recompensa más cercana.


In [11]:
# Experimento C: mayor factor de descuento
grid_C = WarehouseMDP()
grid_C.gamma = 0.99

V_C, n_C = value_iteration(grid_C)
pi_C = extract_policy(grid_C, V_C)

print("=== EXPERIMENTO C ===")
print("gamma:", grid_C.gamma)
print("Iteraciones:", n_C)
print("Valor de START:", round(V_C[grid_C.start], 4))
print("Acción desde START:", ARROWS[pi_C[grid_C.start]])
print_policy(grid_C, pi_C)


=== EXPERIMENTO C ===
gamma: 0.99
Iteraciones: 23
Valor de START: -1.4561
Acción desde START: →
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 


**Interpretación C:** `gamma = 0.99` da más peso a recompensas lejanas que `gamma = 0.9`. La política puede cambiar si esa diferencia altera qué ruta tiene mayor retorno esperado. La comprobación correcta es comparar los valores y las flechas calculados por el notebook.
